# 04 — MLflow Judges & LLM Evaluation

**UI tabs:** Evaluation runs · Judges

| What | API | UI Tab |
|---|---|---|
| Run batch evaluation | `mlflow.genai.evaluate()` | Evaluation runs |
| Auto-score every trace | `judge.register().start()` | Judges |

> Start the MLflow server first: `mlflow server --host 127.0.0.1 --port 5000`

In [ ]:
!pip install mlflow google-genai pandas litellm --quiet

In [ ]:
import os
import mlflow
from google import genai
from google.genai import types
from mlflow.genai import scorer
from mlflow.genai.scorers import Correctness, Guidelines, ScorerSamplingConfig

os.environ["GOOGLE_API_KEY"] = "YOUR_GOOGLE_API_KEY_HERE"
os.environ["GEMINI_API_KEY"] = os.environ["GOOGLE_API_KEY"]  # litellm uses this
client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("04-MLflow-Judges")
print("MLflow", mlflow.__version__, "ready")

## Step 1 — Evaluation dataset + predict function

In [ ]:
eval_dataset = [
    {
        "inputs":       {"question": "What is MLflow?"},
        "expectations": {"expected_response": "MLflow is an open-source platform for managing the ML lifecycle."},
    },
    {
        "inputs":       {"question": "What is experiment tracking?"},
        "expectations": {"expected_response": "Experiment tracking records parameters, metrics and outputs for reproducibility."},
    },
    {
        "inputs":       {"question": "What is a model registry?"},
        "expectations": {"expected_response": "A model registry is a central store for versioning and managing ML models."},
    },
]

# predict_fn receives the keys from inputs as kwargs
def predict(question: str) -> str:
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=[question],
        config=types.GenerateContentConfig(system_instruction="Answer in 1-2 sentences.")
    )
    return (response.text or "").strip()

## Step 2 — Scorers

Two types from the docs:
- **Built-in LLM judge**:  — uses Gemini to check factual accuracy
- **Custom code scorer**:  — plain Python, no LLM needed

In [ ]:
# Built-in: Correctness uses Gemini as the judge model (via litellm)
correctness = Correctness(model="gemini:/gemini-2.5-flash")

# Custom code scorer — from the docs @scorer pattern
@scorer
def is_concise(outputs: str) -> bool:
    """Passes if answer is 30 words or fewer."""
    return len((outputs or "").split()) <= 30

print("Scorers ready")

## Step 3 — Batch evaluation → Evaluation runs tab

In [ ]:
# Runs predict() for each row, scores with each scorer,
# logs everything as an Evaluation Run in the UI
results = mlflow.genai.evaluate(
    data=eval_dataset,
    predict_fn=predict,
    scorers=[correctness, is_concise],
)
print("Done — check Evaluation runs tab")

## Step 4 — Register judge → Judges tab

Registering makes the judge run **automatically on every new trace** — continuous evaluation.

In [ ]:
# From docs: judge.register(name=...).start(sampling_config=...)
# sample_rate=1.0 means score 100% of traces
registered = correctness.register(name="gemini_correctness")
registered.start(sampling_config=ScorerSamplingConfig(sample_rate=1.0))

print("Judge registered — refresh the Judges tab to see it")

## MLflow UI — What to explore

**Next →** `05_mlflow_datasets.ipynb`